# Capítulo 4 — Umidade do ar e balanço de energia

**Curso:** Agrometeorologia Operacional com Python
**Prof. Dr. Fabrício Correia de Oliveira** — UTFPR, Campus Santa Helena

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/agrometeorologia/blob/main/curso/04_umidade_energia.ipynb)

> Pré-requisito: Capítulos 1 a 3.

---


## 4.1 Motivação

A umidade do ar determina quanto potencial de evaporação existe: ar seco "puxa" mais água
da planta e do solo do que ar já saturado. As variáveis de umidade que vamos calcular aqui
(pressão de saturação, pressão parcial de vapor, declividade da curva de pressão de vapor)
são exatamente as entradas que faltam para fechar a equação de Penman-Monteith no
**Capítulo 5** — este capítulo é a ponte entre radiação (Cap. 2) e evapotranspiração (Cap. 5).


## 4.2 Fórmulas

**Pressão de saturação de vapor** (válida para $T_{max}$, $T_{min}$ ou $T_{med}$):
$$e_s(T) = 0{,}6108 \cdot 10^{\frac{7{,}5\,T}{237{,}3+T}}$$

**Pressão de saturação média:**
$$\bar{e}_s = \frac{e_s(T_{max}) + e_s(T_{min})}{2}$$

**Pressão parcial de vapor** (a partir da umidade relativa média, $UR$ em %):
$$e_a = \frac{\bar{e}_s \times UR}{100}$$

**Declividade da curva de pressão de vapor** (usando $T_{med}$):
$$s = \frac{4098 \cdot e_s(T_{med})}{(T_{med} + 237{,}3)^2}$$

**Balanço de energia e razão de Bowen** (revisão do Capítulo 2):
$$R_n = H + LE + G \ (G \approx 0) \qquad \beta = \frac{H}{LE} \implies LE = \frac{R_n}{1+\beta}$$


## 4.3 Do papel ao código

A `agrometeorologiapy` já implementa `es_tetens` (pressão de saturação), `ea_umidade`
(pressão parcial de vapor) e `declive_pressao_vapor` (declividade da curva de pressão de
vapor) no módulo `amp.umidade` — seguindo exatamente as fórmulas acima. Vamos usá-las
diretamente; só `pressao_saturacao_media` (a média simples entre `es(Tmax)` e `es(Tmin)`)
fica aqui, por não ter equivalente pronto na biblioteca.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import agrometeorologiapy as amp


def pressao_saturacao_media(Tmax: float, Tmin: float) -> float:
    """ēs — média entre a pressão de saturação em Tmax e em Tmin (kPa)."""
    return (amp.es_tetens(Tmax) + amp.es_tetens(Tmin)) / 2

## 4.4 Atividade guiada — reproduzindo o exemplo de Palotina-PR

Dados diários: `Tmax = 34,0 °C`, `Tmin = 20,0 °C`, `Tmed = 27,0 °C`, `UR = 65%`. A apostila
chega a `ēs ≈ 3,829 kPa`, `ea ≈ 2,489 kPa` e `s ≈ 0,2091 kPa °C⁻¹` — esses três valores
alimentam diretamente o cálculo de ETo por Penman-Monteith no Capítulo 5.


In [ ]:
Tmax, Tmin, Tmed, UR = 34.0, 20.0, 27.0, 65

es_media = pressao_saturacao_media(Tmax, Tmin)
ea = amp.ea_umidade(es_media, UR)
s = amp.declive_pressao_vapor(Tmed)

print(f"ēs = {es_media:.3f} kPa")
print(f"ea = {ea:.3f} kPa")
print(f"s  = {s:.4f} kPa/°C")
print(f"Déficit de saturação (ēs - ea) = {es_media - ea:.3f} kPa")

## 4.5 Aplicando em dados reais

Vamos calcular o déficit de saturação de vapor (`ēs - ea`) — o principal "motor" da
evapotranspiração — para toda a série de Santa Helena-PR baixada da NASA POWER, e visualizar
sua variação ao longo do ano.


In [ ]:
import requests

LAT, LON = -24.86, -54.33
url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
    "parameters": "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,WS2M",
    "community": "AG",
    "longitude": LON,
    "latitude": LAT,
    "start": "20230101",
    "end": "20231231",
    "format": "JSON",
}
resposta = requests.get(url, params=params, timeout=60)
resposta.raise_for_status()
propriedades = resposta.json()["properties"]["parameter"]

df_clima = pd.DataFrame(propriedades)
df_clima.index = pd.to_datetime(df_clima.index, format="%Y%m%d")
df_clima.index.name = "data"
df_clima = df_clima.replace(-999, np.nan)

df_clima["es_media"] = df_clima.apply(lambda r: pressao_saturacao_media(r["T2M_MAX"], r["T2M_MIN"]), axis=1)
df_clima["ea"] = amp.ea_umidade(df_clima["es_media"], df_clima["RH2M"])
df_clima["deficit_saturacao"] = df_clima["es_media"] - df_clima["ea"]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_clima.index, df_clima["deficit_saturacao"], color="darkorange")
ax.set_ylabel("Déficit de saturação (kPa)")
ax.set_title("Santa Helena-PR — déficit de saturação de vapor (2023)")
plt.tight_layout()
plt.show()

## 4.6 Desafio

1. Identifique o mês com maior déficit médio de saturação e o mês com menor. O que isso
   sugere sobre a demanda evaporativa da atmosfera em cada estação do ano?
2. Calcule `s` (declividade da curva de pressão de vapor) para toda a série e verifique:
   `s` cresce ou decresce com a temperatura? Plote `s` contra `T2M` para visualizar a relação.


In [ ]:
# Espaço para o desafio — escreva seu código aqui


## 4.7 Checkpoint

Antes de seguir para o **Capítulo 5 — Evapotranspiração**, você deve ter:

- [ ] reproduzido o exemplo de Palotina-PR com os mesmos valores da apostila;
- [ ] usado `amp.es_tetens`, `amp.ea_umidade` e `amp.declive_pressao_vapor` testadas e
      funcionando;
- [ ] calculado o déficit de saturação para uma série real de um ano inteiro.

Estas funções, junto com `amp.irradiancia_extraterrestre` e `amp.saldo_radiacao` do
Capítulo 2, são tudo que falta para montar a equação completa de Penman-Monteith no
próximo capítulo.